In [1]:
import numpy as np  
from sklearn.datasets import make_classification 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [21]:
x,y=make_classification(n_samples=1000,n_features=10,n_informative=2,n_redundant=8,weights=[0.9,0.1],flip_y=0,random_state=42)
np.unique(y,return_counts=True)

(array([0, 1]), array([900, 100]))

In [22]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,stratify=y,test_size=0.3)


In [23]:
lr=LogisticRegression(C=1,solver='lbfgs')
lr.fit(x_train,y_train)
y_pred=lr.predict(x_test)
report=classification_report(y_test,y_pred)
print(report)


              precision    recall  f1-score   support

           0       0.95      0.97      0.96       270
           1       0.62      0.50      0.56        30

    accuracy                           0.92       300
   macro avg       0.79      0.73      0.76       300
weighted avg       0.91      0.92      0.92       300



In [24]:
xgb_clf=XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(x_train,y_train)
y_pred_xgb=xgb_clf.predict(x_test)
report=classification_report(y_test,y_pred_xgb)
print(report)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



In [25]:
rf_clf=RandomForestClassifier(n_estimators=30,max_depth=3)
rf_clf.fit(x_train,y_train)
y_pred_rf=rf_clf.predict(x_test)
report=classification_report(y_test,y_pred_rf)
print(report)

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       270
           1       0.95      0.67      0.78        30

    accuracy                           0.96       300
   macro avg       0.96      0.83      0.88       300
weighted avg       0.96      0.96      0.96       300



In [26]:
from imblearn.combine import SMOTETomek
smt=SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(x_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [27]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(x_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



In [28]:
models = [
    (
        "Logistic Regression", 
        LogisticRegression(C=1, solver='liblinear'), 
        (x_train, y_train),
        (x_test, y_test)
    ),
    (
        "Random Forest", 
        RandomForestClassifier(n_estimators=30, max_depth=3), 
        (x_train, y_train),
        (x_test, y_test)
    ),
    (
        "XGBClassifier",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (x_train, y_train),
        (x_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train_res, y_train_res),
        (x_test, y_test)
    )
]

In [29]:
reports = []

for model_name, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [11]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669725,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.91326105087573,
   'support': 300.0}},
 {'0': {'precision': 0.9676258992805755,
   'recall': 0.9962962962962963,
   'f1-score': 0.9817518248175182,
   'support': 270.0},
  '1': {'precision': 0.9545454545454546,
   'recall': 0.7,
   'f1-score': 0.8076923076923077,
   'support': 30.0},
  'accuracy': 0.9666666666666667,
  'macro avg': {'precision': 0.961085676913015,
   'recall': 0.8481481481481481,
   'f1-score': 0.8947220662549129,
   'support': 300.0},
  'weighted avg': {'precision': 0.9663

In [12]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [16]:
mlflow.set_experiment("anamoly detection")
mlflow.set_tracking_uri(uri=" http://127.0.0.1:5000")
for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("model",model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/07/19 10:14:38 INFO mlflow.tracking.fluent: Experiment with name 'anamoly detection' does not exist. Creating a new experiment.
2026/07/19 10:14:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/19 10:14:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at:  http://127.0.0.1:5000/#/experiments/2/runs/98230bb717dc40efb0f2cad97f23e02a
🧪 View experiment at:  http://127.0.0.1:5000/#/experiments/2


2026/07/19 10:14:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at:  http://127.0.0.1:5000/#/experiments/2/runs/edbe57d27a504a8480c2949be1a4f43e
🧪 View experiment at:  http://127.0.0.1:5000/#/experiments/2


2026/07/19 10:15:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at:  http://127.0.0.1:5000/#/experiments/2/runs/fe20698a38ce4b66baef9647f9ea0e2e
🧪 View experiment at:  http://127.0.0.1:5000/#/experiments/2
🏃 View run XGBClassifier With SMOTE at:  http://127.0.0.1:5000/#/experiments/2/runs/c974e5e115024b5789c2093d8b9b6c85
🧪 View experiment at:  http://127.0.0.1:5000/#/experiments/2


In [20]:
run_id="c974e5e115024b5789c2093d8b9b6c85"
model_name="XGBClassifier With SMOTE"
model_uri = f'runs:/{run_id}/model'
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'XGBClassifier With SMOTE' already exists. Creating a new version of this model...
2026/07/19 10:35:57 WARNING mlflow.tracking._model_registry.fluent: Run with id c974e5e115024b5789c2093d8b9b6c85 has no artifacts at artifact path 'model', registering model based on models:/m-f0dd0f3b17814c3c814e1e2961879e30 instead
2026/07/19 10:35:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBClassifier With SMOTE, version 1
Created version '1' of model 'XGBClassifier With SMOTE'.


🏃 View run XGBClassifier With SMOTE at:  http://127.0.0.1:5000/#/experiments/2/runs/c974e5e115024b5789c2093d8b9b6c85
🧪 View experiment at:  http://127.0.0.1:5000/#/experiments/2


In [23]:
model_version = 1
model_uri = f"models:/{model_name}@challenger"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [24]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [25]:
current_model_uri = f"models:/{model_name}@challenger"
production_model_name = "anomaly-detection-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGBClassifier With SMOTE' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1784440934035, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1784440934035, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='c974e5e115024b5789c2093d8b9b6c85', run_link='', source='models:/XGBClassifier With SMOTE/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [26]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [13]:
import dagshub
dagshub.init(repo_owner='rajireddy1582004', repo_name='learn_mlflow', mlflow=True)



❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f72314c3-d00a-4453-a637-2f45090fc152&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a88e782f440816608a14f4f6b4cdc5949291c92a8060ff2613542681160b9a2d




Accessing as rajireddy1582004

Initialized MLflow to track repo "rajireddy1582004/learn_mlflow"

Repository rajireddy1582004/learn_mlflow initialized!

In [31]:

mlflow.set_experiment("Anamoly detection")
for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("model",model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/07/20 18:02:21 INFO mlflow.tracking.fluent: Experiment with name 'Anamoly detection' does not exist. Creating a new experiment.
2026/07/20 18:02:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: https://dagshub.com/rajireddy1582004/learn_mlflow.mlflow/#/experiments/1/runs/1b39766087474f60934b562972c2d6db
🧪 View experiment at: https://dagshub.com/rajireddy1582004/learn_mlflow.mlflow/#/experiments/1


2026/07/20 18:02:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: https://dagshub.com/rajireddy1582004/learn_mlflow.mlflow/#/experiments/1/runs/a8e2aa8a53844595a73d2ae33b47ecda
🧪 View experiment at: https://dagshub.com/rajireddy1582004/learn_mlflow.mlflow/#/experiments/1


2026/07/20 18:03:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: https://dagshub.com/rajireddy1582004/learn_mlflow.mlflow/#/experiments/1/runs/d951578e809440bfad7c6328d252750a
🧪 View experiment at: https://dagshub.com/rajireddy1582004/learn_mlflow.mlflow/#/experiments/1


2026/07/20 18:03:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: https://dagshub.com/rajireddy1582004/learn_mlflow.mlflow/#/experiments/1/runs/3a83d8ffcd9d4b4d8ccbd7b024442391
🧪 View experiment at: https://dagshub.com/rajireddy1582004/learn_mlflow.mlflow/#/experiments/1
